# 210 · Hahn echo timing — solution

**Level:** medium · **Topics:** dsl, sections, timing, Gen2 · **Type:** fix the bugs

Solution of [exercise.ipynb](exercise.ipynb). The three bugs:

1. **`play_after="second_x90"`** – compilation error. `name` is only a prefix: the real section `uid` is unique (for example `second_x90_0`). Pass the section object instead.
2. **`x180` inside a section with `length=tau`** – silent. Sections are left-aligned by default, so the pulse plays at the *start* of the window and the remaining time is padding.
3. **Empty `second_delay` section** – silent. A section only occupies the signals used inside it. An empty section uses none, so the drive line does not wait.

In [ ]:
from laboneq.simple import *
from laboneq.contrib.example_helpers.plotting.plot_helpers import plot_simulation

## Device setup

One SHFQC+ with one qubit: a drive line and a readout line.

📖 [Device setup](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/index.html)

In [ ]:
setup = DeviceSetup("exercise_setup")
setup.add_dataserver(host="localhost", port="8004")
setup.add_instruments(
    SHFQC(uid="shfqc", address="DEV12001", device_options="SHFQC/PLUS/QC2CH")
)
setup.add_connections(
    "shfqc",
    create_connection(to_signal="q0/drive", ports="SGCHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/measure", ports="QACHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/acquire", ports="QACHANNELS/0/INPUT"),
)

## Calibration

📖 [Calibrating signals](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/02_logical_signals/concepts/00_calibrating_signals.html)

In [ ]:
calibration = Calibration()
calibration["q0/drive"] = SignalCalibration(
    oscillator=Oscillator(frequency=100e6, modulation_type=ModulationType.HARDWARE),
    local_oscillator=Oscillator(frequency=5.0e9),
    range=0,
)
calibration["q0/measure"] = SignalCalibration(
    oscillator=Oscillator(frequency=50e6),
    local_oscillator=Oscillator(frequency=7.0e9),
    range=0,
)
calibration["q0/acquire"] = SignalCalibration(
    oscillator=Oscillator(frequency=50e6),
    local_oscillator=Oscillator(frequency=7.0e9),
    range=0,
)
setup.set_calibration(calibration)

## Session

📖 [Session](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/01_session/concepts/index.html)

In [ ]:
session = Session(setup)
session.connect(do_emulation=True)

## Pulses

📖 [Pulse library](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/tutorials/01_pulse_library.html)

In [ ]:
x90 = pulse_library.gaussian(length=100e-9, amplitude=0.5)
x180 = pulse_library.gaussian(length=100e-9, amplitude=1.0)
readout_pulse = pulse_library.const(length=1e-6, amplitude=0.5)

tau = 1e-6  # wait time before and after the x180 pulse

## Experiment (fixed)

📖 [Sections and pulses](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/index.html) ·
[Timing rules](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/01_timing_rules.html) ·
[Section tutorial](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/tutorials/00_section_tutorial.html)

In [ ]:
@dsl.experiment(signals=["drive", "measure", "acquire"])
def hahn_echo():
    with dsl.acquire_loop_rt(count=16):
        with dsl.section(name="first_x90"):
            dsl.play("drive", x90)

        # Fix 2 and 3: explicit delays on the drive line, before and after x180
        with dsl.section(name="echo"):
            dsl.delay("drive", time=tau)
            dsl.play("drive", x180)
            dsl.delay("drive", time=tau)

        with dsl.section(name="second_x90") as second_x90:
            dsl.play("drive", x90)

        # Fix 1: pass the section object, not its name
        with dsl.section(name="readout", play_after=second_x90):
            dsl.measure(
                measure_signal="measure",
                acquire_signal="acquire",
                measure_pulse=readout_pulse,
                integration_kernel=readout_pulse,
                handle="echo",
                reset_delay=1e-6,
            )

In [ ]:
experiment = hahn_echo()
experiment.set_signal_map(
    {
        "drive": "q0/drive",
        "measure": "q0/measure",
        "acquire": "q0/acquire",
    }
)

## Compile and run

📖 [Experiment](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/05_experiment/concepts/index.html)

In [ ]:
compiled = session.compile(experiment)
results = session.run(compiled)

## Check the timing

Plot the simulated output of the first shot.

📖 [Output simulator](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/10_advanced_topics/tutorials/00_output_simulator.html)

In [ ]:
plot_simulation(compiled, start_time=0, length=4e-6, signals=["drive", "measure"])

**Expected result** (τ = 1 µs, pulses of 100 ns):

| Pulse | Starts at |
| --- | --- |
| first `x90` | 0 µs |
| `x180` | 1.1 µs |
| second `x90` | 2.2 µs |
| readout | 2.3 µs |

## Answers

1. The error says that section `second_x90` is not defined. With `dsl.section(name=...)` the `uid` gets a unique suffix, so the string does not match. Use `with dsl.section(...) as second_x90:` and `play_after=second_x90`.
2. Right after the first `x90`, at 0.1 µs. `length` only sets the size of the section; with the default `SectionAlignment.LEFT` the content starts at the beginning. `SectionAlignment.RIGHT` would move the pulse to the *end* of the window, which is still not the same timing as `delay` + pulse.
3. At 1.1 µs, immediately after the `echo_delay` section. The empty section runs in parallel and delays nothing. It would need `dsl.reserve("drive")` or `dsl.delay("drive", time=tau)` inside.
4. The simplest fix is one section with `delay` – `play` – `delay` on the drive line, as above.

**Takeaway:** do not use the `length` of a section to position a pulse. Use `dsl.delay` on the signal line.

📖 [Timing rules](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/01_timing_rules.html) ·
[Pulse sheet viewer](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/02_pulse_sheet_viewer.html)